# 1.12 真的更新一次，代價會怎樣？


回到一個旋鈕：w=1，目標是 3，代價 `(w-3)²=4`。剛才我們會算梯度 -4，現在才真正改參數。

更新規則是 `新參數=舊參數−學習率×梯度`。**學習率**是每次步子的比例；用 0.1，新 w 為 `1−0.1×(-4)=1.4`。再用新位置重算，代價是 `(1.4-3)²=2.56`。這叫**梯度下降**：希望下降的是代價，參數本身可能增加。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-01-update.svg")))

圖把三個動作分開。梯度 -4 和參數 1 是兩份數字；先有梯度，再按指定學習率把參數改成 1.4，最後才得到新代價。


In [ ]:
import torch

w = torch.tensor(1.0, requires_grad=True)
before = (w - 3).square()
before.backward()
print("更新前", w.item(), before.item(), w.grad.item())
with torch.no_grad():
    w -= 0.1 * w.grad
after = (w - 3).square()
print("更新後", w.item(), after.item())
assert after.item() < before.item()

第一行印更新前的 `1、4、-4`。`w -= 0.1*w.grad` 真正改 w；它放在 `torch.no_grad()` 裡，明確把這次調參數排除在求導記錄之外。離開區塊後，`after` 用新 w 重新算代價，所以第二行約為 `1.4、2.56`。

舊的 `before` 儲存的是原來算出的 4，不會因為 w 改了就自動變 2.56。要觀察更新效果，一定得重新前向計算。

正確方向配太大步子仍可能變差：學習率 2 會讓 w 跳到 9，代價 36。練習先改成 0.6，手算 w=3.4、代價 0.16，再執行；接著改成 2，原本「代價下降」的斷言應失敗。這個反例是在檢查步子，而不是宣稱求導算錯。

同樣規則可更新 [1.6](https://birdhackor.github.io/tiny-perceptron-vlm/1.6.html) 的接字分數。現有小實驗讓一張接字表在 9 篇短文上更新 200 次，平均訓練代價由 3.7733 降到 1.0568。它顯示多次更新改變了猜法；能否接出合適新句，還需要生成與留出材料的檢查。

<details>
<summary>補充與重做</summary>

接字表實驗的資料、命令與未教過短文見 [T.3](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.3)。本節只做一次求導，沒有清梯度的問題；下一節會看重複求導。PyTorch 不允許在一般求導模式下直接改需追蹤的葉參數，拿掉 `no_grad()` 會在原地更新處報錯。

</details>
